# Co-tenant covert storage channels — RunPod runner

A long-running matrix on a rented GPU, built so that **losing the pod costs at most one episode**.

Three layers of durability, in increasing order of paranoia:

1. **Per-episode fsync.** `run.py` appends each episode to a JSONL log and `fsync`s it before the
   next one starts. A kill loses the episode in flight and nothing else.
2. **Network volume** (`/workspace`). Survives a pod stop/start, so a restarted pod resumes.
3. **Private Hugging Face dataset repo.** Survives the pod being *destroyed* or reclaimed, which
   is the normal end of a Community Cloud instance. Results are pushed periodically during the
   run and pulled back at startup.

Resume is not a special mode: run this notebook top to bottom again and completed episodes are
skipped, because `done` is rebuilt from the append-only log rather than from a state file that a
hard kill can truncate.

## 1. GPU and precision

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "No GPU visible. Check the pod has one attached."
props = torch.cuda.get_device_properties(0)
GIB = props.total_memory / 1024**3
print(f"device: {props.name}  {GIB:.1f} GiB  compute {props.major}.{props.minor}")
print(f"torch {torch.__version__}, cuda {torch.version.cuda}, bf16: {torch.cuda.is_bf16_supported()}")

# The harness picks quantisation from VRAM: below 24 GiB it uses 4-bit NF4, at or above it keeps
# bf16. A 24 GB card reports ~23.6 GiB and therefore falls on the wrong side of that line, even
# though bf16 fits: Qwen3-8B is ~16.4 GB of weights plus ~1.2 GB of KV cache at 8k context.
# 4-bit there would be SLOWER, not safer -- every matmul pays a dequantisation cost.
FORCE_BF16 = 20.0 <= GIB < 24.0
if FORCE_BF16:
    print(f"\n  {GIB:.1f} GiB: bf16 fits but sits under the 24 GiB auto-threshold.")
    print("  Set ARS_LOAD_4BIT=0 in the config cell (already done below) to keep bf16.")
    print("  If this OOMs, lower ARS_NUM_CTX before reaching for 4-bit.")

## 2. Paths and the network volume

Everything durable lives under `/workspace`, which is where a RunPod **Network Volume** mounts.
If no volume is attached this still runs, but a destroyed pod takes the results with it — the
Hugging Face sync in §3 is what makes that survivable either way.

In [ ]:
import pathlib, shutil, os

ROOT    = pathlib.Path("/workspace")
WORK    = ROOT / "ars"            # unpacked code
RESULTS = ROOT / "results"        # episode logs, calibration, runs
RESULTS.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)

total, used, free = shutil.disk_usage(ROOT)
print(f"{ROOT}: {free/2**30:.1f} GiB free of {total/2**30:.1f} GiB")

# A network volume is a separate mount; if /workspace is on the container's overlay filesystem it
# is ephemeral. This is a warning rather than an assert because the HF sync covers the gap.
with open("/proc/mounts") as fh:
    mounts = fh.read()
on_volume = any(line.split()[1] == "/workspace" for line in mounts.splitlines()
                if len(line.split()) > 1)
print("network volume mounted at /workspace:", on_volume)
if not on_volume:
    print("  WARNING: /workspace is container-local. Results survive only via the HF sync below.")

## 3. Hugging Face auth and the private results repo

Paste a **write** token. It is read with `getpass`, so it is not echoed and does not end up in the
notebook's saved output. Set `HF_TOKEN` in the pod's environment to skip the prompt entirely.

The repo is created **private**. It holds the code bundle and every result artifact.

In [ ]:
import os, getpass
from huggingface_hub import HfApi, create_repo, whoami

tok = os.environ.get("HF_TOKEN") or getpass.getpass("HF write token (hidden): ").strip()
os.environ["HF_TOKEN"] = tok
os.environ["HUGGING_FACE_HUB_TOKEN"] = tok

api = HfApi(token=tok)
me = whoami(token=tok)
print("authenticated as:", me["name"])

# A read token authenticates fine and then fails at the first upload, an hour into the run,
# so the permission is checked now rather than discovered later. The shape of whoami()'s reply
# varies across hub versions, hence the defensive walk.
def _role(info):
    try:
        return ((info.get("auth") or {}).get("accessToken") or {}).get("role")
    except Exception:
        return None

scope = _role(me)
print("token role:", scope or "(not reported)")
if scope == "read":
    raise SystemExit("This is a READ token. Uploads will fail -- create a WRITE token.")

REPO_ID = f"{me['name']}/ars-covert-channel-results"
create_repo(REPO_ID, repo_type="dataset", private=True, exist_ok=True, token=tok)
print("private dataset repo ready:", REPO_ID)

## 4. Pull anything a previous pod already produced

This is the cross-pod half of resume. Episodes already on the Hub are restored to `/workspace`,
and the runner then skips them by name.

In [ ]:
from huggingface_hub import snapshot_download

try:
    got = snapshot_download(REPO_ID, repo_type="dataset", token=tok,
                            local_dir=str(ROOT / "_hf"), allow_patterns=["results/**"])
    src = pathlib.Path(got) / "results"
    n = 0
    if src.exists():
        for p in src.rglob("*"):
            if p.is_file():
                dst = RESULTS / p.relative_to(src)
                dst.parent.mkdir(parents=True, exist_ok=True)
                # Never overwrite a longer local log with a shorter remote one: the local file may
                # already contain episodes this pod appended since the last sync.
                if not dst.exists() or p.stat().st_size > dst.stat().st_size:
                    shutil.copy2(p, dst)
                    n += 1
    print(f"restored {n} file(s) from the Hub")
except Exception as e:
    print(f"nothing to restore ({type(e).__name__}) -- first run on this repo")

eps = list(RESULTS.rglob("episodes.jsonl"))
done = sum(sum(1 for line in p.read_text(encoding='utf-8').splitlines() if line.strip())
           for p in eps)
print(f"{len(eps)} episode log(s) present, {done} episode(s) already recorded")

## 5. Virtual environment

Created with `--system-site-packages` **on purpose**. RunPod's PyTorch images ship a torch build
matched to the pod's CUDA driver; reinstalling torch into an isolated venv is the single most
common way this breaks, and it costs a 2.5 GB download to do it. Inheriting torch and isolating
everything above it gives reproducibility where it matters without touching the one package whose
version is tied to the hardware.

Every later cell runs its work through `PY`, the venv interpreter — the notebook kernel itself is
left alone.

In [ ]:
import subprocess, sys, time

VENV = ROOT / "venv"
PY   = str(VENV / "bin" / "python")

if not (VENV / "bin" / "python").exists():
    print("creating venv (inheriting the image's torch) ...")
    subprocess.run([sys.executable, "-m", "venv", "--system-site-packages", str(VENV)],
                   check=True)

def stream(cmd, cwd=None):
    '''Run a child process and echo its output here, line by line, with an elapsed clock.

    Jupyter redirects Python-level sys.stdout but not fd 1, so a child process's output
    otherwise vanishes into the pod log and the cell looks frozen. capture_output is no better:
    it withholds everything until exit, which over a multi-hour run is the same as silence.
    '''
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(f"[{time.time()-t0:6.0f}s] {line}", end="", flush=True)
    proc.wait()
    print(f"[{time.time()-t0:6.0f}s] exit {proc.returncode}", flush=True)
    return proc.returncode

PKGS = ["transformers>=4.44", "accelerate", "huggingface_hub", "safetensors", "sentencepiece"]
if GIB < 24 and not FORCE_BF16:
    PKGS.append("bitsandbytes")     # only needed where 4-bit is actually used

stream([PY, "-m", "pip", "install", "-q", "--upgrade", *PKGS])
stream([PY, "-c",
        "import torch, transformers, huggingface_hub as h; "
        "print('venv torch', torch.__version__, 'cuda', torch.version.cuda, "
        "'| transformers', transformers.__version__, '| hub', h.__version__); "
        "print('cuda visible to venv:', torch.cuda.is_available())"])

## 6. Code bundle

Looks for `ars-code.zip` on the Hub first, so a fresh pod needs nothing but this notebook and the
token. If the zip is not there yet, upload it once from your machine (`/workspace/ars-code.zip`)
and this cell will push it for next time.

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download, upload_file

ZIP = ROOT / "ars-code.zip"

if not ZIP.exists():
    try:
        p = hf_hub_download(REPO_ID, "ars-code.zip", repo_type="dataset", token=tok)
        shutil.copy2(p, ZIP)
        print("code bundle pulled from the Hub")
    except Exception:
        raise SystemExit(
            f"No code bundle. Upload ars-code.zip to {ZIP} (Jupyter file browser), then re-run "
            "this cell -- it will be pushed to the Hub so later pods find it automatically.")
else:
    upload_file(path_or_fileobj=str(ZIP), path_in_repo="ars-code.zip",
                repo_id=REPO_ID, repo_type="dataset", token=tok)
    print("code bundle uploaded to the Hub for future pods")

if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)
print("unpacked:", sorted(p.name for p in WORK.iterdir()))

## 7. Self-check — seven suites, no GPU, no network

In [ ]:
rc = stream([PY, "check.py"], cwd=str(WORK))
assert rc == 0, "check.py failed -- the bundle is incomplete or stale; re-package and re-upload"

## 8. Configuration

`CONDITIONS` is the cost lever. Δ is defined on `open` vs `wipe` and nothing else, and the
closure-ladder result the other arms speak to is already measured without a GPU by
`src/capacity.py`. The list is condition-major, so an interrupted long run still leaves `open`
and `wipe` complete.

In [ ]:
MODEL_ALIAS = "qwen8-hf"

env = dict(os.environ)
env["ARS_THINK_BUDGET"]  = "6144"
env["ARS_ANSWER_BUDGET"] = "512"
env["ARS_PACE_SECONDS"]  = "0"
if FORCE_BF16:
    env["ARS_LOAD_4BIT"] = "0"     # see 1: a 24 GB card fits bf16 and is slower in 4-bit

PROBE_BUDGETS = [12, 16, 9]
CALIB_SEEDS   = "0,1,2"

CONDITIONS  = "open,wipe"      # ~40 episodes. Full set: add no_substrate,legit,open_lowsalience,content,dirname
GENERATIONS = 10               # the pre-registered n per arm; do not cut this to save time
AGENTS      = 2
MAX_TURNS   = 30

os.environ.update({k: v for k, v in env.items() if k.startswith("ARS_")})
print(f"{MODEL_ALIAS} | conditions={CONDITIONS} | gens={GENERATIONS} x agents={AGENTS}")
print(f"think={env['ARS_THINK_BUDGET']} answer={env['ARS_ANSWER_BUDGET']} "
      f"4bit={'forced off' if FORCE_BF16 else 'auto'}")

## 9. Background sync to the Hub

A daemon thread pushes `results/` every few minutes while the matrix runs, so losing the pod
costs at most one sync interval rather than the whole run. Uploads are incremental — unchanged
files are not re-sent.

In [ ]:
import threading
from huggingface_hub import upload_folder

SYNC_EVERY_S = 300
_stop = threading.Event()
_sync_lock = threading.Lock()

def sync_now(msg="checkpoint"):
    with _sync_lock:
        # upload_folder on an empty tree is a no-op at best and an error at worst; there is
        # nothing to push before the first episode lands.
        if not any(RESULTS.rglob("*")):
            return True
        try:
            upload_folder(folder_path=str(RESULTS), path_in_repo="results",
                          repo_id=REPO_ID, repo_type="dataset", token=tok,
                          commit_message=msg)
            return True
        except Exception as e:
            # A failed sync must never kill the run: the local logs are still durable and the
            # next interval retries.
            print(f"  [sync] failed ({type(e).__name__}: {e}) -- retrying next interval",
                  flush=True)
            return False

def _loop():
    while not _stop.wait(SYNC_EVERY_S):
        if sync_now(f"auto-sync {time.strftime('%Y-%m-%d %H:%M:%SZ', time.gmtime())}"):
            print(f"  [sync] pushed at {time.strftime('%H:%M:%S')}", flush=True)

_syncer = threading.Thread(target=_loop, daemon=True)
_syncer.start()
print(f"background sync every {SYNC_EVERY_S}s -> {REPO_ID}")

## 10. Capability gates — can this model do the task at all?

In [ ]:
rc = stream([PY, "-u", "colab/gate.py", "--model", MODEL_ALIAS, "--samples", "3"], cwd=str(WORK))
assert rc == 0, f"capability gates failed (exit {rc}). A model that cannot do the task makes a null Delta meaningless."

## 11. Calibration gate

The pre-registered stopping condition: solo success must be strictly between 0 and 1. At the floor
or the ceiling the design has no power and Δ would be zero by construction — reporting that is the
honest outcome, not something to engineer past.

In [ ]:
import json

CALIB = RESULTS / "calib-runpod"
n_seeds = len(CALIB_SEEDS.split(","))

def rate_at(budget):
    eps = []
    for f in (CALIB / f"b{budget}").glob("*/episodes.jsonl"):
        for line in f.read_text(encoding="utf-8").splitlines():
            if line.strip():
                eps.append(json.loads(line))
    clean = [e for e in eps if not e.get("api_error")]
    if not clean:
        return None, 0, len(eps) - len(clean)
    return (sum(1 for e in clean if e.get("success")) / len(clean),
            len(clean), len(eps) - len(clean))

MAX_PROBES = None
for budget in PROBE_BUDGETS:
    r, n, errs = rate_at(budget)
    if r is None or n < n_seeds:
        print(f"\n=== calibrating probe budget {budget} ===", flush=True)
        stream([PY, "-u", "runner/calibrate.py", "--models", MODEL_ALIAS,
                "--budgets", str(budget), "--seeds", CALIB_SEEDS, "--generations", "1",
                "--agents", "1", "--max-turns", str(MAX_TURNS), "--outdir", str(CALIB)],
               cwd=str(WORK))
        sync_now(f"calibration budget {budget}")
        r, n, errs = rate_at(budget)
    else:
        print(f"\n=== budget {budget}: reusing {n} episode(s) from an earlier pod ===")

    print(f"budget {budget}: solo success {r} over n={n} clean ({errs} api-error excluded)")
    if r is None:
        continue
    if 0.0 < r < 1.0:
        MAX_PROBES = budget
        print(f"GATE PASSED at probe budget {budget} (solo success {r:.2f})")
        break
    print(f"budget {budget}: at the {'floor' if r == 0.0 else 'ceiling'}; trying the next")

assert MAX_PROBES is not None, (
    "GATE NOT PASSED at any candidate budget. Not running the matrix: with the baseline arm "
    "pinned the design has no power and Delta would be zero by construction.")
print(f"\nusing --max-probes {MAX_PROBES}")

## 12. The matrix

The long one. Safe to interrupt: re-run this cell, or the whole notebook on a new pod, and
completed episodes are skipped.

In [ ]:
RUNS = RESULTS / "runs"
RUNS.mkdir(parents=True, exist_ok=True)

t0 = time.time()
try:
    rc = stream([PY, "-u", "runner/run.py",
                 "--conditions", CONDITIONS, "--models", MODEL_ALIAS, "--seeds", "0",
                 "--generations", str(GENERATIONS), "--agents", str(AGENTS),
                 "--max-probes", str(MAX_PROBES), "--max-turns", str(MAX_TURNS),
                 "--outdir", str(RUNS)],
                cwd=str(WORK))
    print("run.py exit:", rc)
finally:
    # Runs on interrupt too, so a Ctrl-C or a kernel stop still pushes what was completed.
    sync_now("matrix checkpoint")
    print(f"elapsed {(time.time()-t0)/3600:.2f} h; results synced")

## 13. Status and Δ so far

In [ ]:
rc = stream([PY, "analyze/verify.py"], cwd=str(WORK))
stream([PY, "analyze/scorecard.py"], cwd=str(WORK))

## 14. Final sync and shut the pod down

Stop the sync thread, push everything, and confirm the Hub has it. **Check the printed file count
before you destroy the pod.**

In [ ]:
_stop.set()
ok = sync_now("final")
print("final sync:", "ok" if ok else "FAILED -- do not destroy this pod yet")

from huggingface_hub import list_repo_files
files = [f for f in list_repo_files(REPO_ID, repo_type="dataset", token=tok)
         if f.startswith("results/")]
print(f"{len(files)} result file(s) on the Hub at {REPO_ID}")
for f in sorted(files)[:15]:
    print("  ", f)
if len(files) > 15:
    print(f"   ... and {len(files)-15} more")
print("\nPull them locally with:")
print(f"  huggingface-cli download {REPO_ID} --repo-type dataset --local-dir ./results-from-runpod")